In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from distorted import step1, step2, fix1, evaluation, prompts
load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_seed52_ratio0.3_distorted_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10

fix_repetition = 3
current_df = df.copy() 

for iteration in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f">>> ITERATION {iteration + 1} START")
    print(f"{'='*30}")
    activity_list = current_df['activity'].unique().tolist()
    activity_list_json = json.dumps(activity_list, indent=4, ensure_ascii=False)
    
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP1, 
                             prompts.USER_PROMPT_DISTORTED_STEP1)

    if not res_s1['original_activity']:
        print(f">>> No more distorted candidates found in iteration {iteration + 1}. Breaking loop.")
        break
    print(f"Candidates found ({len(res_s1['original_activity'])} total): {res_s1['original_activity'][:5]} ...")

    clean_labels = res_s1['original_activity']
    act_freq_dict = current_df['activity'].value_counts().to_dict()
    act_freq_dict_json = json.dumps(act_freq_dict, indent=4, ensure_ascii=False)

    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, clean_labels, 
                             act_freq_dict_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP2, 
                             prompts.USER_PROMPT_DISTORTED_STEP2)

    print(f"\n>>> Cluster Preview (Total: {len(res_s2)} groups)")
    for i, (clean, variants) in enumerate(res_s2.items()):
        if i >= 3: break  
        print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s2)

    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)

    print(f">>> Iteration {iteration + 1} complete.")





>>> ITERATION 1 START
>>> Running Step 1  with 10 repetitions...
Candidates found (10 total): ['Check for completeness', 'Deliver card', 'Make decision', 'Notify accept', 'Perform checks'] ...
>>> Running Step 2  with 10 repetitions...

>>> Cluster Preview (Total: 10 groups)
  - Check for completeness: ['CHeCk foR comPlEteneSs', 'Ceck for completeness', 'Cehck for completeness'] ... (+22 more)
  - Deliver card: ['DElivEr cArd', 'DEliver card', 'DeLiver card'] ... (+21 more)
  - Make decision: [',ake decision', 'MaRke decision', 'Mak decision'] ... (+20 more)
>>> Starting Activity Correction...
>>> Correction complete. Total rows processed: 53972

>>> EVALUATION

>>> Starting Evaluation...
------------------------------
EVALUATION SUMMARY
------------------------------
eval_status
Correct (TN)    46657
Correct (TP)     6993
Missed (FN)       322
------------------------------
Recall:    95.60%
Precision: 100.00%
------------------------------

>>> Error Samples (Top 5):
    original_ac

In [2]:
LOG_NAME = "pub_seed52_ratio0.3_distorted_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10

fix_repetition = 3
current_df = df.copy() 

for iteration in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f">>> ITERATION {iteration + 1} START")
    print(f"{'='*30}")
    activity_list = current_df['activity'].unique().tolist()
    activity_list_json = json.dumps(activity_list, indent=4, ensure_ascii=False)
    
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP1, 
                             prompts.USER_PROMPT_DISTORTED_STEP1)

    if not res_s1['original_activity']:
        print(f">>> No more distorted candidates found in iteration {iteration + 1}. Breaking loop.")
        break
    print(f"Candidates found ({len(res_s1['original_activity'])} total): {res_s1['original_activity'][:5]} ...")

    clean_labels = res_s1['original_activity']
    act_freq_dict = current_df['activity'].value_counts().to_dict()
    act_freq_dict_json = json.dumps(act_freq_dict, indent=4, ensure_ascii=False)

    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, clean_labels, 
                             act_freq_dict_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP2, 
                             prompts.USER_PROMPT_DISTORTED_STEP2)

    print(f"\n>>> Cluster Preview (Total: {len(res_s2)} groups)")
    for i, (clean, variants) in enumerate(res_s2.items()):
        if i >= 3: break  
        print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s2)

    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)

    print(f">>> Iteration {iteration + 1} complete.")





>>> ITERATION 1 START
>>> Running Step 1  with 10 repetitions...
Candidates found (9 total): ['Bring drinks', 'Bring food', 'Deliver to customer', 'Prepare main course', 'Prepare starter'] ...
>>> Running Step 2  with 10 repetitions...

>>> Cluster Preview (Total: 9 groups)
  - Bring drinks: ['Birng drinks', 'BrIng drinKs', 'Brig drinks'] ... (+19 more)
  - Bring food: ['BrJing food', 'BriMng food', 'BriNg fOod'] ... (+21 more)
  - Deliver to customer: ['DElIver tO custOmEr', 'De,iver to customer', 'DeLiver to CUsTomeR'] ... (+21 more)
>>> Starting Activity Correction...
>>> Correction complete. Total rows processed: 56524

>>> EVALUATION

>>> Starting Evaluation...
------------------------------
EVALUATION SUMMARY
------------------------------
eval_status
Correct (TN)    49048
Correct (TP)     5849
Missed (FN)      1627
------------------------------
Recall:    78.24%
Precision: 100.00%
------------------------------

>>> Error Samples (Top 5):
       original_activity             a